# 📈 Quantiles in Statistics & Data Science: SAT Score Analysis (R Skeleton)

## Expanded Project: Theory, Visualization, Simulation & ML Applications in R

### Learning Objectives (R version)
- Understand quantiles, deciles, quartiles, and percentiles deeply
- Use base R `quantile()` and tidyverse effectively
- Visualize distributions with `ggplot2` + decile reference lines
- Interpret a target SAT score (1350) across different school distributions
- Build a flexible simulation to test different scores and school selectivities
- Apply quantiles in data analysis and machine learning workflows in R
- Practice both base R and modern tidyverse/ggplot2 approaches

**Scenario**: You scored 1350 on the SAT. Analyze three fake universities' accepted student distributions using **deciles** to decide where to apply.

---
Fill in the `# TODO` sections. Use the Solution R notebook for reference.


## 1. Theory: Quantiles – Core Concepts

**Quantiles** divide a dataset into groups of equal size.

- **n quantiles** → n+1 groups
- **Median** = 2-quantile (50th percentile)
- **Quartiles** (Q1=25%, Q2=median, Q3=75%) → IQR is robust spread measure
- **Deciles** (10-quantiles) → used in this project
- **Percentiles** (100-quantiles) → very common in education/testing

### Why Quantiles Matter
- Robust to outliers (unlike mean)
- Excellent for understanding skewness and spread
- Percentile ranks are intuitive ("top 10%", "bottom 30%")
- Foundation for many statistical tests and ML techniques

In R:
- `quantile(x, probs = c(0.1, 0.5, 0.9))`
- `quantile()` has 9 different algorithms (`type = 1` to `type = 9`). Default is usually fine for most work.


## 2. Data Generation & Visualization with Deciles

In [ ]:
library(ggplot2)
library(dplyr)
library(patchwork)   # for combining plots nicely

set.seed(42)
n <- 500

school_one   <- rnorm(n, mean = 1250, sd = 120) |> pmin(1600) |> pmax(400)
school_two   <- rnorm(n, mean = 1380, sd = 90)  |> pmin(1600) |> pmax(400)
school_three <- rnorm(n, mean = 1480, sd = 70)  |> pmin(1600) |> pmax(400)

deciles_one   <- quantile(school_one,   probs = seq(0.1, 0.9, by = 0.1))
deciles_two   <- quantile(school_two,   probs = seq(0.1, 0.9, by = 0.1))
deciles_three <- quantile(school_three, probs = seq(0.1, 0.9, by = 0.1))

# Function to create nice plot with deciles
plot_school <- function(data, deciles, title, target = 1350) {
  df <- data.frame(score = data)
  dec_df <- data.frame(decile = paste0("D", 1:9), value = deciles)
  
  p <- ggplot(df, aes(x = score)) +
    geom_histogram(bins = 30, fill = "#4FC3F7", color = "#0277BD", alpha = 0.7) +
    geom_vline(xintercept = deciles, color = "red", linetype = "dashed", alpha = 0.7) +
    geom_vline(xintercept = target, color = "green", linewidth = 1.3, linetype = "solid") +
    labs(title = title, x = "SAT Score", y = "Number of Accepted Students") +
    theme_minimal() +
    theme(plot.title = element_text(face = "bold"))
  
  # Add decile labels
  for (i in seq_along(deciles)) {
    p <- p + annotate("text", x = deciles[i], y = Inf, label = paste0("D", i),
                      vjust = 1.5, hjust = 1.1, color = "darkred", size = 3)
  }
  return(p)
}

p1 <- plot_school(school_one,   deciles_one,   "School One (Less Selective)")
p2 <- plot_school(school_two,   deciles_two,   "School Two (Moderately Selective)")
p3 <- plot_school(school_three, deciles_three, "School Three (Highly Selective)")

(p1 / p2 / p3)

## 3. Interpreting Your Score (SAT = 1350)

**School One (Less Selective)**  
1350 is high in the distribution — you would likely be in the **top 20–30%** of accepted students. This is generally a **Safety** school.

**School Two (Moderately Selective)**  
Around the middle to upper-middle. Likely **4th to 6th decile**. This is a **Match** school.

**School Three (Highly Selective)**  
Lower in the distribution — probably **2nd or 3rd decile**. This would be a **Reach** school and potentially unrealistic without very strong supporting application materials.

### Simple Decision Rule
- **≥ 7th decile** → Safety  
- **3rd – 6th decile** → Match  
- **≤ 2nd decile** → Reach (consider carefully)


## 4. 🎮 Simulation: Change Your Score or School Parameters

Modify the values below and re-run the chunk.


In [ ]:
# ============== SIMULATION PARAMETERS ==============
YOUR_SCORE       <- 1350
SCHOOL_ONE_MEAN  <- 1250
SCHOOL_TWO_MEAN  <- 1380
SCHOOL_THREE_MEAN<- 1480
SCHOOL_SD        <- 100
N                <- 500
# ========================================================

set.seed(123)
s1 <- rnorm(N, SCHOOL_ONE_MEAN,  SCHOOL_SD) |> pmin(1600) |> pmax(400)
s2 <- rnorm(N, SCHOOL_TWO_MEAN,  SCHOOL_SD) |> pmin(1600) |> pmax(400)
s3 <- rnorm(N, SCHOOL_THREE_MEAN, SCHOOL_SD) |> pmin(1600) |> pmax(400)

dec1 <- quantile(s1, seq(0.1, 0.9, 0.1))
dec2 <- quantile(s2, seq(0.1, 0.9, 0.1))
dec3 <- quantile(s3, seq(0.1, 0.9, 0.1))

get_decile <- function(score, deciles) {
  which.min(score > deciles)   # returns 1-9, or 10 if above all
}

cat("Your SAT Score:", YOUR_SCORE, "
")
cat("School One decile  :", get_decile(YOUR_SCORE, dec1), "
")
cat("School Two decile  :", get_decile(YOUR_SCORE, dec2), "
")
cat("School Three decile:", get_decile(YOUR_SCORE, dec3), "
")

# Visual feedback for School Two
df2 <- data.frame(score = s2)
ggplot(df2, aes(x = score)) +
  geom_histogram(bins = 30, fill = "#81C784", color = "#2E7D32", alpha = 0.7) +
  geom_vline(xintercept = dec2, color = "red", linetype = "dashed") +
  geom_vline(xintercept = YOUR_SCORE, color = "green", linewidth = 1.5) +
  labs(title = paste("School Two - Your Score =", YOUR_SCORE),
       subtitle = "Red lines = Deciles") +
  theme_minimal()

## 5. Alternate Approaches in R

**Base R style** (very compatible):
```r
hist(school_two, breaks = 30, col = "lightblue")
abline(v = deciles_two, col = "red", lty = 2)
abline(v = 1350, col = "green", lwd = 2)
```

**Tidyverse + ggplot2** (recommended for most modern work) — shown in the main visualization above.

**Exact percentile rank**:
```r
ecdf(school_two)(1350) * 100   # gives percentile
```


## 6. Advanced & Machine Learning Uses of Quantiles in R

1. **Robust Scaling**
   ```r
   # Using IQR (based on quantiles)
   robust_scale <- function(x) (x - median(x)) / IQR(x)
   ```

2. **Creating Decile / Percentile Features**
   ```r
   df$decile <- ntile(df$score, 10)        # dplyr::ntile
   df$percentile <- percent_rank(df$score)
   ```

3. **Quantile Regression** (very powerful)
   ```r
   library(quantreg)
   model <- rq(y ~ x, data = df, tau = 0.9)   # 90th percentile model
   summary(model)
   ```

4. **Outlier Detection**
   ```r
   lower <- quantile(x, 0.01)
   upper <- quantile(x, 0.99)
   outliers <- x[x < lower | x > upper]
   ```

Quantiles are foundational in robust statistics and modern machine learning pipelines in R.


## 🗺️ Analysis Flowchart

```mermaid
flowchart TD
    A[Generate or Load Score Data] --> B[Compute Deciles with quantile()]
    B --> C[Visualize with ggplot2 + geom_vline]
    C --> D[Locate Target Score in Distribution]
    D --> E[Classify as Safety / Match / Reach]
    E --> F[Simulation: Change score or parameters]
    F --> G[Advanced: Quantile features, quantile regression]
    G --> H[Final Insights & Recommendations]
```


## ✏️ More Practice Exercises (R)

1. Compute Q1, Q2 (median), Q3 and IQR for all three schools using `quantile()`.
2. Write a function `analyze_score(score, data, school_name)` that returns decile + recommendation.
3. Create side-by-side boxplots of the three schools using `ggplot2`.
4. Add a fourth school with very low variance and re-analyze 1350.
5. Use `dplyr::ntile()` to create a decile feature and explore its distribution.
6. (Advanced) Install `quantreg` and fit a simple quantile regression model.


## ✅ Key Takeaways

- Quantiles provide a robust, intuitive way to understand distributions.
- Deciles are particularly useful for dividing data into 10 clear groups.
- A score of 1350 has very different meaning depending on school selectivity.
- Simulation helps you understand how sensitive recommendations are.
- In R, combine base R `quantile()` with `ggplot2` for professional analysis.
- Quantiles power many advanced techniques: robust scaling, feature engineering, and quantile regression.

This R version gives you the same powerful analytical framework as the Python version, using tools that are very common in statistics-heavy teams and academia.
